# Country Missingness Scoring

Identifies countries systematically absent from data they should have.
Classifies each (country, year) as dissolved, microstate, failed, degraded, reporting, or strong.

**Phase 1: Model Definition**

In [ ]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

In [ ]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs")

## Run Pipeline

In [ ]:
result = run_country_missingness(df, meta_df)

## Country Profiles

Dissolved states, microstates, and temporal spans.

In [ ]:
# Dissolved states
filter(r -> r.is_dissolved, result.profiles)

In [ ]:
# Microstates
filter(r -> r.is_microstate, result.profiles)

## Status Distribution

How many country-years fall in each category?

In [ ]:
sort(combine(groupby(result.status, :country_status), nrow => :count), :count, rev=true)

## Failed & Degraded Countries

Which active countries are losing data coverage?

In [ ]:
# Countries ever classified as failed (excluding dissolved/micro)
failed = filter(r -> r.country_status == "failed", result.status)
failed_countries = unique(failed.ident_ccode)
println("Countries with 'failed' years: $(length(failed_countries))")

# Show their trajectory: status by decade
for ccode in failed_countries[1:min(10, length(failed_countries))]
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    alpha = filter(r -> r.ident_ccode == ccode, result.profiles).ident_ccodealp[1]
    name = filter(r -> r.ident_ccode == ccode, result.profiles).ident_cname[1]
    println("\n  $alpha $name:")
    for decade_start in [1990, 2000, 2010, 2020]
        decade = filter(r -> decade_start <= r.ident_year < decade_start + 10, rows)
        if nrow(decade) > 0
            statuses = unique(decade.country_status)
            avg_cov = round(mean(decade.global_coverage_pct) * 100, digits=1)
            println("    $(decade_start)s: $(join(statuses, "/")) ($(avg_cov)% avg coverage)")
        end
    end
end

## Revised Slug Penetration

Slugs that gain penetration when failed states are excluded from the denominator.

In [ ]:
# Top gainers
first(result.penetration, 20)

In [ ]:
# How many slugs cross the 95% threshold with revised denominator?
original_global = count(r -> r.original_penetration >= 0.95, eachrow(result.penetration))
revised_global = count(r -> r.revised_penetration >= 0.95, eachrow(result.penetration))
println("Slugs ≥95% penetration:")
println("  Original denominator: $original_global")
println("  Revised denominator:  $revised_global")
println("  New globals:          $(revised_global - original_global)")

## Save Flags

In [ ]:
# CSV.write("data/country_missingness_flags.csv", result.flags)
# println("\u2705 Saved country_missingness_flags.csv")